In [1]:
import sys
sys.path.append('../src/')

In [2]:
from constants import INPT_VARS, EXTRA_VARS, OUT_VARS, GLOBAL_COMBINED_STATS
import hydra
from hydra.utils import instantiate
from pathlib import Path
import os
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import logging

from utils.data_utils import (
    get_wet_mask,
    get_train_test_ranges,
    gen_data_in_test,
    gen_data_out_test,
    data_CNN_Lateral,
    data_CNN_Dynamic,
    gen_data_025_lateral,
    gen_data_global_new,
)
from utils.eval_utils import (
    generate_model_rollout,
    compute_mean,
    compute_var,
    compute_corrs_area,
    compute_rmse,
    compute_corrs,
    compute_KE,
    compute_time_spec,
    compute_ACC,
    compute_nino34,
    compute_amo,
    gen_KE_spectrum,
    gen_KE,
    gen_KE_range,
    gen_value_range,
    gen_enstrophy_spectrum,
    gen_enstrophy,
    compute_corrs_single,
    compute_ACC_single,
    compute_RMSE_single,
    compute_mean_single,
)
from utils.subgrid_utils import get_area_tensor
from utils.climate_utils import compute_laplacian_wet
from utils.plot_utils import (
    plot_short_time_stats,
    plot_long_time_stats,
    plot_map,
    plot_error_map,
    plot_both_error_map,
    plot_metrics_KE_spectrum,
    plot_metrics_KE,
    plot_metrics_enstrophy_spectrum,
    plot_metrics_entrophy,
    plot_metrics_corr,
    plot_metrics_rmse,
    plot_metrics_acc,
    plot_metrics_mean,
    plot_metrics_pdf,
    get_initial_snapshot_fig,
    plot_region_based_metric,
    plot_diff_map,
)

import numpy as np
import torch
import xarray as xr
import copy

In [3]:
class Eval:
    def __init__(self, args):
        # Getting input, extra input and output
        self.inputs = INPT_VARS[args.exp_num_in]
        self.extra_in = EXTRA_VARS[args.exp_num_extra]
        self.outputs = OUT_VARS[args.exp_num_out]

        self.str_in = "".join([i + "_" for i in self.inputs])
        self.str_ext = "".join([i + "_" for i in self.extra_in])
        self.str_out = "".join([i + "_" for i in self.outputs])

        print("inputs: " + self.str_in)
        print("extra inputs: " + self.str_ext)
        print("outputs: " + self.str_out)

        self.N_atm = len(self.extra_in)  # Number of atmosphere variables
        self.N_in = len(self.inputs)
        if args.lateral:
            self.N_extra = (
                self.N_atm + self.N_in
            )  # Number of atmosphere variables + Lateral boundary variables
        else:
            self.N_extra = self.N_atm  # Number of atmosphere variables
        self.N_out = len(self.outputs)

        self.num_in = int((args.hist + 1) * self.N_in + self.N_extra)

        print("Number of inputs: ", self.num_in)  # 3 (ocean speeds + ocean temp)(t) +
        # 3 (atm wind stresses + atm temp)(t) +
        # 3 (boundary ocean speeds + boundary ocean temp)(t) -> 3 (ocean speeds + ocean temp)(t+1)
        print("Number of outputs: ", self.N_out)  # 3

        # Post-fix strings
        self.str_train = (
            "steps_"
            + str(args.steps)
            + "_"
            + args.train_region
            + "_Test_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "_out"
            + self.str_out
            + "N_train_4000"
            + "_Lateral_Data_025_no_smooth"
        )
        self.str_save = (
            "steps_"
            + str(args.steps)
            + "_"
            + args.train_region
            + "_"
            + args.region
            + "_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "N_samples_"
            + str(args.N_samples)
        )
        self.post_model_name = (
            "Train_" + args.train_region
            + "_Test_" + args.region
            + "_Test_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "_out"
            + self.str_in
            + "N_train_"
            + str(args.N_samples)
            + "_Lateral_Data_025_no_smooth"
        )
        self.post_pred_name = (
            args.region
            + "_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "N_samples_"
            + str(args.N_samples)
        )

        # Getting start and end indices of train and test
        s_train, e_train, e_test = get_train_test_ranges(
            args.N_samples, args.N_val, args.lag, args.hist, args.interval
        )

        # Saving data
        print("Getting inputs")
        if "global_1" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(self.inputs, self.extra_in, self.outputs, args.lag)
        elif "global_2x" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(self.inputs, self.extra_in, self.outputs, args.lag, run_type ="2x")
        elif "global_4x" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(self.inputs, self.extra_in, self.outputs, args.lag, run_type ="4x")
        else:
            raise NotImplementedError

        print("Calculating mask tensors")
        self.wet, self.wet_nan = get_wet_mask(inputs, "cpu")
        self.wet_bool = np.array(self.wet.cpu()).astype(bool)
        wet_lap = compute_laplacian_wet(self.wet_nan, 4) # hardcoded
        wet_lap = xr.where(wet_lap == 0, 1, np.nan)
        self.wet_lap = np.nan_to_num(wet_lap)
        print("Wet resolution:", self.wet.shape)

        self.time_vec = inputs[0].time.data

        self.time_test = self.time_vec[e_test : (e_test + args.lag * args.N_test)]

        print("Loading Train data")
        train_data = torch.load(
                    Path(args.data_dir) / "train_data_cnn_{0}.pt".format(self.str_train),
                    map_location=torch.device("cpu"),
                )
        self.train_data = train_data
    
        if args.save_test_data:
            print("Saving data")
            data_in_test = gen_data_in_test(
                0, e_test, args.N_test, args.lag, args.hist, inputs, extra_in
            )
            data_out_test = gen_data_out_test(
                0, e_test, args.N_test, args.lag, args.hist, outputs
            )
            if "global" in args.region:
                norm_vals = train_data.norm_vals
                if "combined" in args.train_region:
                    assert len(norm_vals) == len(GLOBAL_COMBINED_STATS) and all(np.array_equal(norm_vals[k], GLOBAL_COMBINED_STATS[k]) for k in norm_vals)
                self.test_data = data_CNN_Dynamic(
                    data_in_test,
                    data_out_test,
                    self.wet.to(device="cpu"),
                    norm_vals,
                    device=args.device,
                )
                # del train_data
            else:
                raise NotImplementedError()
            torch.save(
                self.test_data,
                Path(args.data_dir) / "test_data_cnn_{0}.pt".format(self.str_save),
            )

        else:
            print("Loading test data")
            self.test_data = torch.load(
                Path(args.data_dir) / "test_data_cnn_{0}.pt".format(self.str_save)
            )

        # Model
        print("Loading model " + args.network)
        if "swin" in args.network.lower():
            model = instantiate(
                args.swin,
                in_channels=self.num_in,
                output_channels=self.N_in,
                pretrain_img_size=[*self.test_data[0][0].shape[1:]],
                wet=self.wet.cuda()
            )
        elif "unet" in args.network.lower():
            model = instantiate(
                args.unet, wet=self.wet.cuda()
            )

        full_model_path = args.ckpt_path
        self.full_model_name = args.network + "_" + self.post_model_name
        self.output_channels = model.output_channels

        # from torchinfo import summary
        # # summary(model)
        # i = [torch.zeros(1, 9, 128, 192)] * 2
        # summary(model, input_data=[i], col_names=["output_size", "num_params"], depth=4)
        # import pdb; pdb.set_trace()

        model = model.to(args.device)
        if isinstance(full_model_path, list):
            self.model = []
            for model_path in full_model_path:
                model.load_state_dict(
                    torch.load(full_model_path, map_location=torch.device(args.device))
                )
                self.model.append(copy.deepcopy(model))
        else:
            model.load_state_dict(
                torch.load(full_model_path, map_location=torch.device(args.device))
            )
            self.model = model

        # Stats
        self.mean_out = self.test_data.norm_vals["m_out"]
        self.std_out = self.test_data.norm_vals["s_out"]
        self.mean_in = self.test_data.norm_vals["m_in"]
        self.std_in = self.test_data.norm_vals["s_in"]

        # clim
        self.clim = None
        if args.save_clim_data:
            print("Saving clim")
            clim = np.zeros((366, *self.wet.shape, 3))
            for i in range(self.N_out):
                clim[:, :, :, i] = (
                    outputs[i].groupby("time.dayofyear").mean("time").data
                )
            torch.save(
                clim,
                Path(args.data_dir) / "clim_cnn_{0}.pt".format(self.str_save),
            )

        else:
            print("Loading clim")
            clim = torch.load(
                Path(args.data_dir) / "clim_cnn_{0}.pt".format(self.str_save)
            )

        self.clim = clim

        # Getting area tensor
        print("Computing area tensor")
        self.grids = xr.open_dataset('/scratch/as15415/Data/CM2x_grids/Grid_New.nc').rename({"dx": "dxu", "dy": "dyu"})

        self.area = torch.from_numpy(self.grids["area_C"].to_numpy()).to(device="cpu")
        self.dx = self.grids["dxu"].to_numpy()
        self.dy = self.grids["dyu"].to_numpy()

        self.Nb = args.Nb
        self.hist = args.hist
        self.lag = args.lag
        self.N_test = args.N_test
        self.N_samples = args.N_samples
        self.output_dir = args.output_dir
        self.region = args.region
        self.steps = args.steps
        self.network = args.model_name_replace
        self.inputs = inputs

        self.pred_region = args.region
        self.pred_names = args.pred_names if args.pred_names else []
        self.pred_paths = args.pred_paths if args.pred_paths else [[]]

        self.JUPYTER_MODE = False

    def send_data_to_cpu(self):
        self.test_data.set_device(device="cpu")

In [4]:
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
import copy
from datetime import datetime
import os

# G1, G1
# ConvNext UNet
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_multiseed".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet",
#         "train_region=global_1",
#         "region=global_1",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=ConvNext UNet Train1Eval1",
#         "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_2x_seed100/convnext/saved_nets/convnextunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt, /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_2x_seed200/convnext/saved_nets/convnextunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# e = Eval(args)

# ConvNext UNet
with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
    args1 = compose(config_name="exp/eval_unet_global", overrides=[
        "output_dir=./temp/{0}_multiseed".format(str(datetime.now())[:10]),
        "model_name_replace=ConvNext UNet",
        "train_region=global_1",
        "region=global_1",
        "run_gen_pred=False", # Multi-Seed Generation
        "network=ConvNext UNet Train1Eval1",
        "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_2x_seed100/convnext/saved_nets/convnextunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt, /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_2x_seed200/convnext/saved_nets/convnextunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
        "pred_names=null",
        "pred_paths=null"
    ])
if not os.path.exists(args1.output_dir):
    os.mkdir(args1.output_dir)

e = Eval(args1)

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Calculating mask tensors
Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading clim
Computing area tensor


In [28]:

# Baseline
network = 'Adam UNet Train1Eval1'
ckpt_paths = ["/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-13-foundation_train_adamunet_global_1/adamunetseed/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt", 
            "/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_2x_seed100/adam/saved_nets/adamunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt", 
              "/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_2x_seed200/adam/saved_nets/adamunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt"]


Loading model Adam UNet Train1Eval1


### Generation

In [30]:
def generate_pred_lateral(e):
    print("Generation Pred begin...")
    ns = 4000
    for rand_ind, model_path in enumerate(e.ckpt_paths):
        print("Random seed: ", rand_ind+1)
        e.model.load_state_dict(
            torch.load(model_path, map_location=torch.device('cuda'))
        )
        
        model_pred = generate_model_rollout(
            e.N_test,
            e.test_data,
            e.model,
            e.hist,
            e.N_in,
            e.N_extra,
            e.Nb,
            e.region,
        )

        print("data_gen")
        da = xr.DataArray(
            data=model_pred,
            dims=["time", "x", "y", "var"],
        )

        da.to_zarr(
            e.pred_model_path
            / (
                "Pred_lateral_Fast_Data_025_"
                + e.post_pred_name
                + "_rand_seed_"
                + str(rand_ind+1)
                + ".zarr"
            ),
            mode="w",
        )
        print(f"Model pred shape {model_pred.shape}")


In [31]:
generate_pred_lateral(e)


Generation Pred begin...
Random seed:  1


RuntimeError: Error(s) in loading state_dict for UNet:
	Missing key(s) in state_dict: "layers.0.skip_module.weight", "layers.0.skip_module.bias", "layers.0.convblock.0.weight", "layers.0.convblock.0.bias", "layers.0.convblock.1.weight", "layers.0.convblock.1.bias", "layers.0.convblock.1.running_mean", "layers.0.convblock.1.running_var", "layers.0.convblock.2.cap", "layers.0.convblock.3.weight", "layers.0.convblock.3.bias", "layers.0.convblock.4.weight", "layers.0.convblock.4.bias", "layers.0.convblock.4.running_mean", "layers.0.convblock.4.running_var", "layers.0.convblock.5.cap", "layers.0.convblock.6.weight", "layers.0.convblock.6.bias", "layers.2.skip_module.weight", "layers.2.skip_module.bias", "layers.2.convblock.0.weight", "layers.2.convblock.0.bias", "layers.2.convblock.1.weight", "layers.2.convblock.1.bias", "layers.2.convblock.1.running_mean", "layers.2.convblock.1.running_var", "layers.2.convblock.2.cap", "layers.2.convblock.3.weight", "layers.2.convblock.3.bias", "layers.2.c

### Loading and Plotting

In [7]:
pred_names = []
pred_paths = []

In [10]:
e.send_data_to_cpu()

def load_seeded_data(Pred_path, e, current_net=False):
    if current_net:
        prefix =  "Pred_lateral_Fast_Data_025_" + e.post_pred_name + "_rand_seed_"
    else:
        prefix = ("Pred_lateral_Fast_Data_025_" 
                    + e.pred_region
                    + "_in_"
                    + e.str_in
                    + "ext_"
                    + "tau_u_tau_v_t_ref_"
                    + "N_samples_"
                    + str(e.N_samples)
                    + "_rand_seed_")

    # model_preds = np.zeros([])
    for rand_seed in range(1,4):
        print("Loading seed ", rand_seed)
        model_pred_net = (
            xr.open_zarr(
                    Path(Pred_path) / (prefix
                    + str(rand_seed)
                    + ".zarr"
                    )
                )
            ).to_array().to_numpy().squeeze()
        
        if rand_seed == 1:
            model_preds = np.repeat(np.expand_dims(np.zeros_like(model_pred_net),axis=0),3,axis=0)
        
        model_preds[rand_seed-1] = model_pred_net

    # model_preds = np.mean(model_preds, axis=0)
    return model_preds

def load_long_data(e):
    print("Load long data...")
    model_pred_net = load_seeded_data(e.pred_model_path, e, True)

    model_pred_saved_nets = []
    for model_pred_path in pred_paths:
        model_pred_saved_nets.append(
            load_seeded_data(model_pred_path, e, False)
        )
    
    return model_pred_net, model_pred_saved_nets


In [11]:
model_pred_net, model_pred_saved_nets = load_long_data(e)

Load long data...
Loading seed  1
Loading seed  2
Loading seed  3


In [12]:
model_pred_net.shape

(3, 3000, 180, 360, 3)

In [18]:
model_pred_temp = model_pred_net

In [19]:
model_pred_net = np.mean(model_pred_net, axis=0)

In [26]:
def compute_corr_map(e, area_flat, pred, truth):
    cor_KE = (area_flat*pred[e.wet_bool].flatten()*truth[e.wet_bool].flatten()).sum()/np.sqrt((area_flat*pred[e.wet_bool].flatten()**2).sum()*(area_flat*truth[e.wet_bool].flatten()**2).sum())
    return cor_KE

def plot_maps(e, model_pred_net, model_pred_saved_nets, start_map=0, N_plot_map=1000, start_error_map=0, N_plot_error_map=1000, long=False):
    ### Long time KE
    print("Getting Mean KE stats...")
    long_KE_net, long_KE_true = gen_KE_range(
        start_map, N_plot_map, e.test_data, model_pred_net
    )
    long_KE_net = long_KE_net.mean(0)
    long_KE_true = long_KE_true.mean(0)

    area_flat = np.array(e.area[e.wet_bool].flatten())
    logging.info(f"Correlation KE {network}: {compute_corr_map(e, area_flat, long_KE_net, long_KE_true)}")


    long_KE_saved = []
    for i, model_pred_saved in enumerate(model_pred_saved_nets):
        long_KE_savedi, _ = gen_KE_range(
            start_map, N_plot_map, e.test_data, model_pred_saved
        )
        long_KE_savedi = long_KE_savedi.mean(0)
        logging.info(f"Correlation KE {e.pred_names[i]}: {compute_corr_map(e, area_flat, long_KE_savedi, long_KE_true)}")
        long_KE_saved.append(long_KE_savedi)

    print("Plotting mean KE...")
    plot_map(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        e.grids,
        e.Nb,
        e.wet_nan,
        long_KE_true,
        long_KE_saved + [long_KE_net],
        "KE",
        e.JUPYTER_MODE
    )

    print("Getting MAE KE stats...")

    long_KE_net, long_KE_true = gen_KE_range(
        start_error_map, N_plot_error_map, e.test_data, model_pred_net
    )
    _, long_KE_train_true = gen_KE(1000, e.train_data, model_pred_net)
    mse_KE_net = long_KE_net.mean(axis=0) - long_KE_true.mean(axis=0) # np.sqrt(((long_KE_net - long_KE_true)**2).mean(axis=0))
    diff_KE = long_KE_train_true.mean(axis=0) - long_KE_true.mean(axis=0)

    long_mse_KE_saved = []
    for model_pred_saved in model_pred_saved_nets:
        long_KE_savedi, _ = gen_KE_range(
            start_error_map, N_plot_error_map, e.test_data, model_pred_saved
        )
        mse_KE_savedi = long_KE_savedi.mean(axis=0) - long_KE_true.mean(axis=0) # np.sqrt(((long_KE_savedi - long_KE_true)**2).mean(axis=0))
        long_mse_KE_saved.append(mse_KE_savedi)
    
    long_KE_true = long_KE_true.mean(0)

    print("Plotting MAE KE")
    plot_error_map(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        e.grids,
        e.Nb,
        e.wet_nan,
        long_KE_true,
        long_mse_KE_saved + [mse_KE_net],
        "KE",
        e.JUPYTER_MODE
    )

    print("Plotting Diff KE")
    plot_diff_map(
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        e.grids,
        e.Nb,
        e.wet_nan,
        long_KE_true,
        diff_KE,
        "KE",
        e.JUPYTER_MODE
    )

    print("Getting Mean Temp Stats")
    long_temp_net, long_temp_true = gen_value_range(
        start_map, N_plot_map, e.test_data, model_pred_net, 2
    )
    long_temp_net = long_temp_net.mean(0)
    long_temp_true = long_temp_true.mean(0)

    logging.info(f"Correlation Temp {network}: {e.compute_corr_map(area_flat, long_temp_net, long_temp_true)}")

    long_temp_saved = []
    for i, model_pred_saved in enumerate(model_pred_saved_nets):
        long_temp_savedi, _ = gen_value_range(
            start_map, N_plot_map, e.test_data, model_pred_saved, 2
        )
        long_temp_savedi = long_temp_savedi.mean(0)
        logging.info(f"Correlation Temp {e.pred_names[i]}: {e.compute_corr_map(area_flat, long_temp_savedi, long_temp_true)}")
        long_temp_saved.append(long_temp_savedi)

    print("Plotting Temp map...")
    plot_map(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        e.grids,
        e.Nb,
        e.wet_nan,
        long_temp_true,
        long_temp_saved + [long_temp_net],
        "TEMP",
        e.JUPYTER_MODE
    )

    
    long_temp_net, long_temp_true = gen_value_range(
        start_error_map, N_plot_error_map, e.test_data, model_pred_net, 2
    )
    _, long_temp_train_true = gen_value_range(0, 1000, e.train_data, model_pred_net, 2)
    # mse_temp_net = np.sqrt(((long_temp_net - long_temp_true)**2).mean(axis=0))
    mse_temp_net = long_temp_net.mean(axis=0) - long_temp_true.mean(axis=0)
    diff_temp = long_temp_train_true.mean(axis=0) - long_temp_true.mean(axis=0)

    

    long_temp_net = long_temp_net.mean(0)

    long_temp_RMSE_saved = []
    for model_pred_saved in model_pred_saved_nets:
        long_temp_savedi, _ = gen_value_range(
            start_error_map, N_plot_error_map, e.test_data, model_pred_saved, 2
        )
        # mse_KE_savedi = np.sqrt(((long_temp_savedi - long_temp_true)**2).mean(axis=0))
        mse_KE_savedi = long_temp_savedi.mean(axis=0) - long_temp_true.mean(axis=0)
        long_temp_RMSE_saved.append(mse_KE_savedi)

    long_temp_true = long_temp_true.mean(0)


    print("Plotting MAE temp map")
    # plot_error_map(
    #     e.pred_names + [network],
    #     e.region if not long else e.region + '_Long_',
    #     e.str_save,
    #     e.output_dir,
    #     e.grids,
    #     e.Nb,
    #     e.wet_nan,
    #     long_temp_true,
    #     long_temp_RMSE_saved + [mse_temp_net],
    #     "TEMP",
    #     e.JUPYTER_MODE
    # )

    plot_both_error_map(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        e.grids,
        e.Nb,
        e.wet_nan,
        long_KE_true,
        long_mse_KE_saved + [mse_KE_net],
        long_temp_true,
        long_temp_RMSE_saved + [mse_temp_net],
        e.JUPYTER_MODE
    )

    print("Plotting Diff Temp")
    plot_diff_map(
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        e.grids,
        e.Nb,
        e.wet_nan,
        long_temp_true,
        diff_temp,
        "TEMP",
        e.JUPYTER_MODE
    )

def plot_timeseries_KE(e, model_pred_net, model_pred_saved_nets, start=0, N_plot=200, N_plot_spec=200, long=False):
    print("Getting KE stats...")
    KE_spec_net, KE_spec_true = gen_KE_spectrum(
        N_plot_spec, e.test_data, model_pred_net, e.grids, e.wet
    )

    KE_spec_saved = []
    for model_pred_saved in model_pred_saved_nets:
        KE_speci, KE_spec_true = gen_KE_spectrum(
            N_plot_spec, e.test_data, model_pred_saved, e.grids, e.wet
        )
        KE_spec_saved.append(KE_speci)

    print("Plotting KE Spectrum...")
    plot_metrics_KE_spectrum(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        KE_spec_true,
        KE_spec_saved + [KE_spec_net],
        e.JUPYTER_MODE
    )

    KE_net, KE_true = compute_KE(
        N_plot, e.test_data, model_pred_net, e.area, e.wet_bool
    )

    KE_saved = []
    for model_pred_saved in model_pred_saved_nets:
        KE_neti, KE_true = compute_KE(
            N_plot, e.test_data, model_pred_saved, e.area, e.wet_bool
        )
        KE_saved.append(KE_neti)

    print("Plotting KE...")
    plot_metrics_KE(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        KE_true,
        KE_saved + [KE_net],
        start,
        N_plot,
        e.JUPYTER_MODE
    )

def plot_timeseries_temperature(e, model_pred_net, model_pred_saved_nets, start=0, N_eval=200, N_eval_ACC=100, long=False):
    ### Spatial matching metrics
    print("Getting Spatial matching stats...")
    T_test = np.array(
        e.test_data[:][1][:, 2] * e.std_out[2] + e.mean_out[2]
    )
    print("Getting Mean...")
    mean_T_net, mean_T_true = compute_mean_single(
        N_eval,
        T_test,
        model_pred_net[:, :, :, 2],
        e.area,
        e.wet_bool,
    )
    mean_T_saved = []
    for model_pred_saved in model_pred_saved_nets:
        mean_T_i, mean_T_true = compute_mean_single(
            N_eval,
            T_test,
            model_pred_saved[:, :, :, 2],
            e.area,
            e.wet_bool,
        )
        mean_T_saved.append(mean_T_i)

    print("Plotting Temperature Mean...")
    plot_metrics_mean(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        mean_T_true,
        mean_T_saved + [mean_T_net],
        start,
        N_eval,
        e.JUPYTER_MODE
    )

def plot_indices(e, model_pred_net, model_pred_saved_nets, long=False):
    print("Getting Nino34...")
    nino_net, nino_true = compute_nino34(
        e.grids,
        e.inputs,
        model_pred_net, 
        e.test_data, 
        e.mean_out,
        e.std_out,
        e.time_test
        )
    nino_saved = []
    for model_pred_saved in model_pred_saved_nets:
        nino_net_i, nino_true = compute_nino34(
                            e.grids,
                            e.inputs,
                            model_pred_saved, 
                            e.test_data, 
                            e.mean_out,
                            e.std_out,
                            e.time_test
                        )
        nino_saved.append(nino_net_i)

    print("Plotting Nino34...")
    plot_region_based_metric(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        nino_true,
        nino_saved + [nino_net],
        e.JUPYTER_MODE,
        mode='nino34'
    )

    print("Getting Amo...")
    amo_net, amo_true = compute_amo(e.grids,
                            e.inputs,
                            model_pred_net,
                            e.test_data, 
                            e.mean_out,
                            e.std_out,
                            e.time_test)
    amo_saved = []
    for model_pred_saved in model_pred_saved_nets:
        amo_net_i, amo_true = compute_amo(e.grids,
                            e.inputs,
                            model_pred_saved,
                            e.test_data, 
                            e.mean_out,
                            e.std_out,
                            e.time_test)
        amo_saved.append(amo_net_i)

    print("Plotting Amo...")
    plot_region_based_metric(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.str_save,
        e.output_dir,
        amo_true,
        amo_saved + [amo_net],
        e.JUPYTER_MODE,
        mode='amo'
    )
    
def plot_pdf(e, model_pred_net, model_pred_saved_nets, start=100, N_days=100, long=False):
    # PDF
    print("Getting PDF stats...")
    pdf = {}
    for ind_plot in range(3):
        true_field = (
            e.test_data[start : start + N_days][1][
                :, ind_plot, e.wet_bool
            ].flatten()
            * e.std_out[ind_plot]
        ) + e.mean_out[ind_plot]
        true_pdf, bins_true = np.histogram(true_field, bins=150, density=True)
        bins_true = (bins_true[1:] + bins_true[:-1]) / 2

        field_net = model_pred_net[
            start : start + N_days, e.wet_bool, ind_plot
        ].flatten()
        pdf_net, bins_net = np.histogram(field_net, bins=150, density=True)
        bins_net = (bins_net[1:] + bins_net[:-1]) / 2

        pdf[ind_plot] = {
            "true_pdf": true_pdf,
            "true": [bins_true, true_pdf],
            network: [bins_net, pdf_net],
        }

        for i, model_pred_saved in enumerate(model_pred_saved_nets):
            field_i = model_pred_saved[
                start : start + N_days, e.wet_bool, ind_plot
            ].flatten()
            pdf_i, bins_i = np.histogram(field_i, bins=150, density=True)
            bins_i = (bins_i[1:] + bins_i[:-1]) / 2

            pdf[ind_plot][e.pred_names[i]] = [bins_i, pdf_i]

    # KE PDF
    long_KE_net, long_KE_true = gen_KE_range(
        start, N_days, e.test_data, model_pred_net
    )

    true_KE_field = long_KE_true[:, e.wet_bool].flatten()
    true_KE_pdf, bins_KE_true = np.histogram(true_KE_field, bins=150, density=True)
    bins_KE_true = (bins_KE_true[1:] + bins_KE_true[:-1]) / 2

    field_KE_net = long_KE_net[:, e.wet_bool].flatten()
    pdf_KE_net, bins_KE_net = np.histogram(field_KE_net, bins=150, density=True)
    bins_KE_net = (bins_KE_net[1:] + bins_KE_net[:-1]) / 2

    pdf["KE"] = {
            "true_pdf": true_KE_pdf,
            "true": [bins_KE_true, true_KE_pdf],
            network: [bins_KE_net, pdf_KE_net],
        }

    for i, model_pred_saved in enumerate(model_pred_saved_nets):
        long_KE_savedi, _ = gen_KE_range(
            start, N_days, e.test_data, model_pred_saved
        )
        field_i = long_KE_savedi[:, e.wet_bool].flatten()
        pdf_i, bins_i = np.histogram(field_i, bins=150, density=True)
        bins_i = (bins_i[1:] + bins_i[:-1]) / 2
        pdf["KE"][e.pred_names[i]] = [bins_i, pdf_i]

    print("Plotting pdf...")
    plot_metrics_pdf(
        e.pred_names + [network],
        e.region if not long else e.region + '_Long_',
        e.output_dir,
        pdf,
        e.JUPYTER_MODE
    )

In [27]:
if args.N_test == 3000:
    plot_maps(e, model_pred_net, model_pred_saved_nets, start_map=1999, N_plot_map=2999, start_error_map=1999, N_plot_error_map=2999, long=True)
    # plot_timeseries_KE(e, model_pred_net, model_pred_saved_nets, start=1999, N_plot=2999, N_plot_spec=1000, long=True)
    # plot_timeseries_enstrophy(e, model_pred_net, model_pred_saved_nets, N_plot=1000, long=True)
    plot_timeseries_temperature(e, model_pred_net, model_pred_saved_nets, start=1999, N_eval=2999, long=True)
    # plot_pdf(e, model_pred_net, model_pred_saved_nets, start=1999, N_days=1000, long=True)
elif args.N_test == 2000:
    plot_maps(e, model_pred_net, model_pred_saved_nets, start_map=999, N_plot_map=1999, start_error_map=999, N_plot_error_map=1999, long=True)
    # plot_timeseries_KE(e, model_pred_net, model_pred_saved_nets, start=999, N_plot=1999, N_plot_spec=1000, long=True)
    # plot_timeseries_enstrophy(e, model_pred_net, model_pred_saved_nets, N_plot=1000, long=True)
    plot_timeseries_temperature(e, model_pred_net, model_pred_saved_nets, start=999, N_eval=1999, long=True)
    # plot_pdf(e, model_pred_net, model_pred_saved_nets, start=999, N_days=1000, long=True)
else:
    raise NotImplementedError()

# plot_indices(e, model_pred_net, model_pred_saved_nets, True)



Getting Mean KE stats...
Plotting mean KE...
Getting MAE KE stats...
Plotting MAE KE
Plotting Diff KE
Getting Mean Temp Stats
Plotting Temp map...
Plotting MAE temp map
Plotting Diff Temp
Getting Spatial matching stats...
Getting Mean...
Plotting Temperature Mean...


<Figure size 1200x500 with 0 Axes>

<Figure size 1200x500 with 0 Axes>

<Figure size 1200x500 with 0 Axes>

<Figure size 1200x500 with 0 Axes>

<Figure size 1200x500 with 0 Axes>

<Figure size 1200x500 with 0 Axes>